In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from typing import Tuple, List, Optional


# =========================
# 1. Reward Model
# =========================
# class RewardModel(nn.Module):
#     def __init__(self, model_name: str = "bert-base-uncased", use_margin_loss: bool = False):
#         super().__init__()
#         # 1. Load a pretrained model as the backbone (shared weights)
#         self.backbone = AutoModel.from_pretrained(model_name)
#         hidden_size = self.backbone.config.hidden_size
        
#         # 2. Add a reward head (outputs a scalar score)
#         self.reward_head = nn.Linear(hidden_size, 1)
#         self.use_margin_loss = use_margin_loss

#     def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
#         # Use the backbone to extract features
#         outputs = self.backbone(input_ids, attention_mask=attention_mask)
#         last_hidden_state = outputs.last_hidden_state  # [batch, seq_len, hidden_size]
        
#         # Take the feature at the [EOS] position (or use mean pooling)
#         eos_embedding = last_hidden_state[:, -1, :]  # use the last token in the sequence
#         # Or: mean_embedding = last_hidden_state.mean(dim=1)
        
#         # Compute reward score
#         reward = self.reward_head(eos_embedding).squeeze(-1)  # [batch]
#         return reward

class RewardModel(nn.Module):
    """
    Reward model built on top of a pretrained transformer backbone.
    The backbone encodes the sequence; a linear head outputs a scalar reward.
    """
    def __init__(self, model_name: str = "bert-base-uncased", use_margin_loss: bool = False):
        super().__init__()
        # Load a pretrained backbone model
        self.backbone = AutoModel.from_pretrained(model_name)
        hidden_size = self.backbone.config.hidden_size

        # Reward head: hidden_size -> 1 scalar
        self.reward_head = nn.Linear(hidden_size, 1)
        self.use_margin_loss = use_margin_loss

    def forward(self, **encodings) -> torch.Tensor:
        """
        Forward pass for the reward model.

        Expects encoder-style inputs, e.g.:
            input_ids, attention_mask, token_type_ids, ...

        We just forward all of them to the backbone.
        """
        outputs = self.backbone(**encodings)  # backbone handles token_type_ids etc.
        last_hidden_state = outputs.last_hidden_state  # [batch, seq_len, hidden_size]

        # Use the last token embedding (similar to EOS)
        eos_embedding = last_hidden_state[:, -1, :]    # [batch, hidden_size]

        # Reward scalar per sequence
        reward = self.reward_head(eos_embedding).squeeze(-1)  # [batch]
        return reward


In [2]:
# =========================
# 2. Preference Dataset
# =========================
class PreferenceDataset(Dataset):
    """
    Loads paired preference data in the format:
        (prompt, chosen_response, rejected_response)
    """
    def __init__(self, data: List[Tuple[str, str, str]], tokenizer: AutoTokenizer, max_length: int = 128):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx) -> Tuple[dict, dict]:
        prompt, chosen, rejected = self.data[idx]
        
        # Encode the 'chosen' sequence:
        # [CLS] prompt [SEP] chosen_response [SEP]
        chosen_enc = self.tokenizer(
            prompt, chosen,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        # Encode the 'rejected' sequence in the same way
        rejected_enc = self.tokenizer(
            prompt, rejected,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        return chosen_enc, rejected_enc


In [3]:
# =========================
# 3. Pairwise Loss
# =========================
def pairwise_loss(
    chosen_rewards: torch.Tensor,
    rejected_rewards: torch.Tensor,
    margins: Optional[torch.Tensor] = None
) -> torch.Tensor:
    """
    Pairwise preference loss used in reward modeling.
    
    Standard form:
        L = -log( sigmoid( r_chosen - r_rejected ) )

    If a margin tensor is provided, we use a margin-based variant:
        L = -log( sigmoid( (r_chosen - r_rejected) - margin ) )

    Args:
        chosen_rewards:   Tensor of shape [batch]
        rejected_rewards: Tensor of shape [batch]
        margins:          Optional tensor of shape [batch] for margin loss

    Returns:
        Scalar loss (mean over batch)
    """
    diff = chosen_rewards - rejected_rewards
    
    # Optional margin loss
    if margins is not None:
        diff = diff - margins

    return -torch.log(torch.sigmoid(diff)).mean()


In [4]:
from transformers import AutoModel, AutoTokenizer

# =========================
# 4. Hyperparameters & Setup
# =========================
model_name = "Qwen/Qwen3-0.6B"   # or the exact HF ID you used originally

backbone = AutoModel.from_pretrained(
    model_name,
    trust_remote_code=True,      # many Qwen models need this
    cache_dir="../02-DeepSeeK/hf_models",
)
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True,
    cache_dir="../02-DeepSeeK/hf_models",
)


In [5]:
# Training hyperparameters
# model_name = "Qwen/Qwen3-0.6B"           # <--- use HF ID, not snapshot hash
# cache_dir = "../02-DeepSeeK/hf_models"   # optional, for local caching
model_name = "bert-base-uncased"

batch_size = 8
grad_accum_steps = 2      # Gradient accumulation (reduces memory usage)
use_amp = True            # Enable mixed-precision training if available
use_margin_loss = True    # Whether to apply margin-based pairwise loss

# Device selection
# or: device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Device selection (CUDA > MPS > CPU)
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Using device:", device)

# Initialize model, tokenizer, optimizer
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    # trust_remote_code=True,      # Qwen often needs this
    # cache_dir=cache_dir,
)
# Initialize model and optimizer
model = RewardModel(model_name, use_margin_loss=use_margin_loss).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-6)

Using device: mps


In [6]:
# Example preference data (replace with your real dataset)
train_data = [
    ("Explain reinforcement learning.",
     "Reinforcement learning is a method where an agent learns through reward signals.",
     "Reinforcement learning is just an algorithm."),

    ("What is Newton's third law?",
     "Action and reaction forces are equal in magnitude and opposite in direction.",
     "Newton's laws are mainly about gravity."),
]

# Dataset & DataLoader
train_dataset = PreferenceDataset(train_data, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

In [8]:
# =========================
# 5. Training Loop
# =========================

# Enable AMP only if we're actually on CUDA
use_amp_eff = use_amp and (device.type == "cuda")

scaler = torch.cuda.amp.GradScaler(enabled=use_amp_eff)
model.train()
num_epochs = 1
for epoch in range(num_epochs):
    total_loss = 0.0
    optimizer.zero_grad()

    for step, batch in enumerate(train_loader):
        chosen_batch, rejected_batch = batch

        # Move batched tokenized inputs to device
        chosen_inputs = {k: v.to(device) for k, v in chosen_batch.items()}
        rejected_inputs = {k: v.to(device) for k, v in rejected_batch.items()}

        # Forward pass with (optional) mixed precision
        with torch.cuda.amp.autocast(enabled=use_amp_eff):
            # Compute reward scores for chosen and rejected responses
            r_chosen = model(**chosen_inputs)      # [batch]
            r_rejected = model(**rejected_inputs)  # [batch]

            # If margin loss is enabled, create a margin tensor.
            # Here we use a constant margin as an example.
            if use_margin_loss:
                margins = torch.full_like(r_chosen, 0.5, device=device)
            else:
                margins = None

            loss = pairwise_loss(r_chosen, r_rejected, margins)
            # If you're doing grad accumulation, scale loss here:
            loss = loss / grad_accum_steps

        # Backprop through scaled loss (for AMP stability)
        if use_amp_eff:
            scaler.scale(loss).backward()
        else:
            loss.backward()

        total_loss += loss.item()  # this is averaged loss per mini-step

        # Gradient accumulation step
        if (step + 1) % grad_accum_steps == 0:
            if use_amp_eff:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            optimizer.zero_grad()

    # Recover the original loss scale before printing
    avg_loss = (total_loss * grad_accum_steps) / len(train_loader)
    print(f"Epoch {epoch + 1}, Loss: {avg_loss:.4f}")

/var/folders/7x/tfwsytqd3yjccjl53cm75j700000gn/T/ipykernel_2074/342712274.py:8: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp_eff)
/var/folders/7x/tfwsytqd3yjccjl53cm75j700000gn/T/ipykernel_2074/342712274.py:23: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp_eff):


ValueError: too many values to unpack (expected 2)

In [ ]:
# =========================
# 6. Save the trained reward model
# =========================

torch.save(model.state_dict(), "reward_model.pt")
print("Saved reward model to reward_model.pt")